# Targeted Positive and Negative Inference Pipeline

Runs inference over raster embedding TIFFs and writes positive predictions as
shapefiles.

Each tile is read, featurized **and scored** inside a worker process, so only
the rows that pass the threshold cross back to the parent. Peak memory is
therefore set by one tile per worker rather than by the size of the region,
which is what makes stride 8 possible.

## 1. Imports and setup

Core geospatial, raster, numerical and model utilities. The custom `cut_chips`
function is imported from the local `../gee/` folder.

In [1]:
from concurrent.futures import ProcessPoolExecutor
from functools import partial
from glob import glob
from pathlib import Path
import json
import multiprocessing as mp
import os
import pickle
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import shapely
from tqdm import tqdm

sys.path.append("../gee")
from tile_utils import cut_chips

# Optional: caps BLAS threads inside each worker (ships with scikit-learn).
try:
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None

/home/ec2-user/miniforge3/envs/earth_genome/lib/python3.13/site-packages/descarteslabs/core/client/__init__.py:28: FutureWarning: Python version 3.13 is not supported yet. You may encounter unexpected errors.
  warnings.warn(msg, FutureWarning)
/home/ec2-user/miniforge3/envs/earth_genome/lib/python3.13/site-packages/descarteslabs/core/compute/function.py:45: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Configuration

Everything that changes between runs lives here.

In [ ]:
# Region to run inference over; matches the embeddings folder name below.
region = "priors_germany_centroids"

# Chip size and stride in pixels. Equal values give non-overlapping chips.
patch_size = 16
stride = 8

# Cap the number of tiles for a quick test run; None runs the whole region.
max_tiles = None

# Worker processes. Try n_workers + 2 if iostat shows the run is I/O-bound.
n_workers = os.cpu_count() or 1

# Keeping negatives puts every patch back in parent memory 
# and will OOM the kernel. Use only on small validation tranches.
keep_negatives = True

# Cast tiles on read. AlphaEarth embeddings are int8-quantized, and numpy
# promotes int8 -> float64 in mean/std: 2x the memory and ~1.45x the time for
# ~1e-6 relative difference in the features. None preserves the source dtype.
read_dtype = "float32"

# GDAL settings applied inside each worker.
# - DISABLE_READDIR_ON_OPEN: skips scanning the 14k-entry tile directory for
#   sidecar files on every open.
# - CACHEMAX (MB): the 5% -of-RAM default would give 4 workers ~1.6 GB each.
# - NUM_THREADS: workers already saturate the cores; internal threads only
#   add contention.
gdal_options = {
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "GDAL_CACHEMAX": "256",
    "GDAL_NUM_THREADS": "1",
}

## 3. Locate raster embedding files

Collect the tiles and read the channel count and dtype from the first one.

In [3]:
# Directory containing the AlphaEarth embedding rasters for this region.
raster_embeddings_path = Path(
    f"/home/ec2-user/work/climate_trace/germany_classifications/data/{region}AlphaEarth"
)

paths = sorted(glob(str(raster_embeddings_path / "*.tif")))

# Fail early if the directory is empty or the path is wrong.
if not paths:
    raise FileNotFoundError(
        f"No .tif files found under {raster_embeddings_path}. "
        "Set raster_embeddings_path to your embeddings directory."
    )

if max_tiles is not None:
    paths = paths[:max_tiles]

# Read once here so feature columns can be named before the parallel pass.
with rasterio.open(paths[0]) as src:
    embedding_dim = src.count
    source_dtype = src.dtypes[0]
    tile_shape = (src.height, src.width)

print(f"{len(paths):,} tiles | {embedding_dim} channels | {source_dtype} | "
      f"{tile_shape[0]}x{tile_shape[1]} px | {n_workers} workers")

29,132 tiles | 64 channels | float32 | 160x160 px | 4 workers


## 4. Define patch-level feature summaries

Each patch is summarized by mean, standard deviation, maximum and minimum
across its spatial dimensions.

Stats are taken on the native `(N, H, W, C)` layout rather than transposing to
`(N, C, H, W)` first. The output is bit-identical and it drops one array copy.

In [4]:
def expert_stats(patches, axis=(1, 2)):
    """Compute summary statistics for raster embedding patches.

    Parameters
    ----------
    patches : ndarray
        Array with shape ``(N, H, W, C)``, where ``N`` is the number of
        patches, ``H`` and ``W`` are patch height and width, and ``C`` is the
        number of embedding channels.
    axis : tuple
        Axes over which to compute summary statistics.

    Returns
    -------
    tuple of ndarray
        Mean, standard deviation, maximum and minimum, each ``(N, C)``.
    """
    return (
        patches.mean(axis=axis),
        patches.std(axis=axis),
        patches.max(axis=axis),
        patches.min(axis=axis),
    )


# Number of statistics returned by expert_stats.
n_stats = len(expert_stats(np.zeros((1, 1, 1, 1))))

# One column per channel per statistic, ordered channel-major (c * n_stats + s)
# to match the reshape in tile_predict. This ordering must match training.
feature_columns = [f"e{i}" for i in range(embedding_dim * n_stats)]

## 5. Per-tile worker: featurize and score

The model is loaded once per worker by the pool initializer, not once per tile.
Each call returns only the patches that pass the threshold, as a probability
array plus WKB geometries — a few KB instead of the ~23 MB of features the tile
generated. Errors come back as values so one bad tile cannot kill the pool.

In [5]:
def init_worker(model_path, metrics_path, gdal_options):
    """Load the model once per worker process and pin its thread usage."""
    global MODEL, THRESHOLD

    # GDAL reads its config options from the environment.
    os.environ.update(gdal_options)

    # Without this, each of the N workers may start its own BLAS thread pool
    # and the processes fight over the same cores.
    if threadpool_limits is not None:
        global _THREAD_LIMITS
        _THREAD_LIMITS = threadpool_limits(limits=1)

    with open(model_path, "rb") as f:
        MODEL = pickle.load(f)
    THRESHOLD = json.loads(Path(metrics_path).read_text())["threshold"]


def tile_predict(path, patch_size, stride, keep_negatives=False, read_dtype=None):
    """Read, featurize and score one tile; return only the rows worth keeping.

    Returns
    -------
    tuple
        ``(path, prob, wkb, crs_wkt, error)``. ``prob`` holds positive-class
        probabilities and ``wkb`` one well-known-binary geometry per kept
        patch. Both are ``None`` for empty tiles, tiles with no positives, and
        failures, in which case ``error`` describes the problem.
    """
    try:
        with rasterio.open(path) as src:
            bounds = src.bounds
            crs = src.crs
            img = src.read(out_dtype=read_dtype) if read_dtype else src.read()

        # Convert from rasterio format (C, H, W) to image format (H, W, C).
        img = np.moveaxis(img, 0, -1)

        # Cut raster into chips and keep corresponding chip geometries.
        patches, geoms = cut_chips(
            img,
            bounds,
            chip_size=patch_size,
            stride=stride,
            crs=crs,
        )
        n_patches = len(patches)
        if n_patches == 0:
            return path, None, None, None, None

        # Stack stats to (N, C, S), then flatten to (N, C * S).
        expert = np.stack(expert_stats(patches), axis=-1)
        _, n_channels, n_stat = expert.shape
        if n_channels * n_stat != len(feature_columns):
            return path, None, None, None, (
                f"expected {len(feature_columns)} features, "
                f"got {n_channels * n_stat}"
            )

        # A DataFrame keeps sklearn's feature-name check happy; wrapping an
        # existing array in one does not copy it.
        features = pd.DataFrame(
            expert.reshape(n_patches, n_channels * n_stat), columns=feature_columns
        )
        prob = MODEL.predict_proba(features)[:, 1]

        keep = np.ones(n_patches, bool) if keep_negatives else prob >= THRESHOLD
        if not keep.any():
            return path, None, None, None, None

        wkb = gpd.GeoSeries(geoms["geometry"]).to_wkb()
        if len(wkb) != n_patches:
            return path, None, None, None, (
                f"geometry/patch mismatch: {len(wkb)} geoms vs {n_patches} patches"
            )

        return path, prob[keep], wkb[keep], crs.to_wkt(), None
    except Exception as exc:
        return path, None, None, None, f"{type(exc).__name__}: {exc}"

## 6. Score all tiles in parallel

`chunksize=1` keeps the workers evenly loaded: at ~20 ms per tile the
per-task overhead is negligible next to the idle time a coarse chunking
leaves at the tail of the run.

In [6]:
model_path = Path("../model/model.pkl")
metrics_path = Path("../model/metrics.json")
threshold = json.loads(metrics_path.read_text())["threshold"]

worker = partial(
    tile_predict,
    patch_size=patch_size,
    stride=stride,
    keep_negatives=keep_negatives,
    read_dtype=read_dtype,
)

prob_parts, geom_parts, crs_seen, failures = [], [], set(), []

# "fork" lets the workers see the functions and feature_columns defined in this
# notebook. To run on macOS or Windows (spawn only), move expert_stats,
# tile_predict and feature_columns into ../gee/tile_utils.py and import them.
ctx = mp.get_context("fork")

with ProcessPoolExecutor(
    max_workers=n_workers,
    mp_context=ctx,
    initializer=init_worker,
    initargs=(model_path, metrics_path, gdal_options),
) as pool:
    results = pool.map(worker, paths, chunksize=1)

    for path, prob, wkb, crs_wkt, error in tqdm(results, total=len(paths)):
        if error is not None:
            failures.append((path, error))
            continue
        if prob is None:
            continue
        prob_parts.append(prob)
        geom_parts.append(wkb)
        crs_seen.add(crs_wkt)

print(f"{len(paths):,} tiles | {len(prob_parts):,} with detections | "
      f"{len(failures):,} skipped")
for path, error in failures[:10]:
    print(f"  skipped {Path(path).name}: {error}")

100%|██████████| 29132/29132 [17:19<00:00, 28.02it/s]


29,132 tiles | 29,132 with detections | 0 skipped


## 7. Combine into one prediction GeoDataFrame

Stack the surviving rows in a single concatenation and rebuild geometries from
WKB, after checking that every tile shares a CRS.

In [7]:
if not prob_parts:
    raise RuntimeError(
        "No rows returned; check the failures above and the threshold."
    )

if len(crs_seen) > 1:
    raise ValueError(
        f"Tiles do not share a CRS ({len(crs_seen)} distinct); reproject first."
    )

pred_prob = np.concatenate(prob_parts)
geometry = gpd.GeoSeries.from_wkb(np.concatenate(geom_parts), crs=crs_seen.pop())

# With keep_negatives=False every row is a positive by construction; the column
# is kept so section 8's `pred == 1` query works either way.
grid_pred = gpd.GeoDataFrame(
    {"pred": (pred_prob >= threshold).astype("uint8"), "pred_prob": pred_prob},
    geometry=geometry,
    crs=geometry.crs,
)

del prob_parts, geom_parts

print(f"{len(grid_pred):,} rows at threshold {threshold}")
print(grid_pred["pred"].value_counts(dropna=False))

10,516,652 rows at threshold 0.45
pred
0    10052956
1      463696
Name: count, dtype: int64


## 8. Save prediction shapefile

Write the prediction geometries and attributes to disk, organized by stride.

In [8]:
output_dir = Path(f"../inference/stride_{stride}/{region}")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "predictions.shp"
grid_pred.to_file(output_path, driver="ESRI Shapefile")

# GeoParquet is faster to write and read, keeps full field names and dtypes,
# and has no 2 GB limit. Worth using if you ever set keep_negatives=True.
# grid_pred.to_parquet(output_dir / "predictions.parquet")

print(f"Wrote {output_path} ({len(grid_pred):,} rows)")

Wrote ../inference/stride_8/priors_germany_centroids/predictions.shp (10,516,652 rows)


## 9. Post process

Use stride 16 as locator and stride 8 for footprint restricted to priors.

In [9]:
# --- Load -----------------------------------------------------------------
priors_path = Path("../data/priors_germany.gpkg")
priors = gpd.read_file(priors_path)

pos_16 = (
    gpd.read_file(Path(f"../inference/stride_{patch_size}/{region}/predictions.shp"))
    .query("pred == 1")
    .reset_index(drop=True)
)
pos_8 = (
    gpd.read_file(Path(f"../inference/stride_{stride}/{region}/predictions.shp"))
    .query("pred == 1")
    .reset_index(drop=True)
)

# A CRS mismatch yields an empty join rather than an error, so check up front.
if pos_8.crs != pos_16.crs:
    raise ValueError(f"stride CRS mismatch: {pos_16.crs} vs {pos_8.crs}")
priors = priors.to_crs(pos_16.crs)

# overlaps|equals is only correct if both layers use the same chip size.
a16, a8 = pos_16.geometry.area.median(), pos_8.geometry.area.median()
if not np.isclose(a16, a8, rtol=0.01):
    raise ValueError(
        f"chip areas differ ({a16:.0f} vs {a8:.0f} m2) - stride-8 chips are nested "
        'inside locators, so use shapely.covered_by instead of overlaps|equals'
    )

def keep_intersecting(gdf, other):
    """Rows of gdf intersecting at least one geometry in other, no duplication."""
    hits = gdf.sjoin(other[["geometry"]], how="inner", predicate="intersects")
    return gdf.loc[hits.index.unique()]


# --- Step 1: stride-16 positives on a prior become the locators ------------
locators = keep_intersecting(pos_16, priors)

# --- Step 2: stride-8 positives on a prior, restricted to those locators ---
cand_8 = keep_intersecting(pos_8, priors)

if locators.empty or cand_8.empty:
    raise ValueError("Nothing survived the prior filter; check the priors extent/CRS.")

# Spatial index gives candidate pairs; the predicates then apply QGIS
# "overlaps or equals" exactly. equals is false for genuine overlaps and
# overlaps is false for identical chips, so the union covers both cases.
pairs = cand_8.sjoin(locators[["geometry"]], how="inner", predicate="intersects")

chip = pairs.geometry.to_numpy()
loc = locators.geometry.to_numpy()[locators.index.get_indexer(pairs["index_right"])]
pairs = pairs.loc[shapely.overlaps(chip, loc) | shapely.equals(chip, loc)]

# n_loc (1-4 on a half-step grid) indicates how centred a detection is.
footprints = cand_8.loc[pairs.index.unique()].assign(
    n_loc=pairs.groupby(level=0).size()
)

print(f"stride 16: {len(pos_16):,} positives -> {len(locators):,} on priors")
print(f"stride 8:  {len(pos_8):,} positives -> {len(cand_8):,} on priors "
      f"-> {len(footprints):,} on locators")
print(footprints["n_loc"].value_counts().sort_index())

# --- Save -----------------------------------------------------------------
post_dir = Path(f"../inference/post/{region}")
post_dir.mkdir(parents=True, exist_ok=True)
locators.to_file(post_dir / "locators.shp", driver="ESRI Shapefile")
footprints.to_file(post_dir / "footprints.shp", driver="ESRI Shapefile")

/tmp/ipykernel_6333/2552720284.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  a16, a8 = pos_16.geometry.area.median(), pos_8.geometry.area.median()
/tmp/ipykernel_6333/2552720284.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  a16, a8 = pos_16.geometry.area.median(), pos_8.geometry.area.median()


stride 16: 124,681 positives -> 77,624 on priors
stride 8:  463,696 positives -> 296,036 on priors -> 278,781 on locators
n_loc
1     105536
2      73097
3      43277
4      27244
5      14329
6       8016
7       3547
8       2078
9        872
10       412
11       185
12        95
13        29
14        27
15        25
16         8
17         4
Name: count, dtype: int64
